In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [2]:
import random

In [3]:
random.seed(42)

In [4]:
tests = ['gc_50_3', 'gc_70_7', 'gc_100_5', 'gc_250_9', 'gc_500_1', 'gc_1000_5']
thresholds = [(8, 6), (20, 17), (21, 16), (95, 78), (18, 16), (124, 100)]

In [18]:
def load_graph(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, m = list(map(int, lines[0].split()))
        edges = list()
        for i in range(m):
            u, v = list(map(int, lines[1 + i].split()))
            edges.append((u, v))

        return n, m, edges

In [25]:
def check_coloring(n, m, edges, color):
    for i in range(m):
        u, v = edges[i]
        if color[u] == color[v]:
            raise Exception("Not correct coloring")
                
    result = max(color)
    return result

In [26]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [27]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, m, edges = load_graph(test)
        start = time.time()
        
        if not use_file:
            coloring = method(n, m, edges)
        else:
            coloring = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_coloring(n, m, edges, coloring)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Сначала напишем жадный алгоритм, который проходится по вершинам в некотором порядке и красит вершину в минимальный возможный цвет. 
Будем пробовать несколько случайных порядков.

In [28]:
!g++ -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [29]:
def greedy_coloring(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        coloring = list(map(int, lines[0].split()))
        return coloring

In [30]:
test_method(greedy_coloring, "greedy", True)

Checking greedy
Execution time: 0.2371 seconds
Target function gc_50_3: 7
Passed gc_50_3: 1
Execution time: 0.2216 seconds
Target function gc_70_7: 20
Passed gc_70_7: 1
Execution time: 0.3095 seconds
Target function gc_100_5: 19
Passed gc_100_5: 1
Execution time: 4.0501 seconds
Target function gc_250_9: 94
Passed gc_250_9: 1
Execution time: 1.4067 seconds
Target function gc_500_1: 18
Passed gc_500_1: 1
Execution time: 29.6082 seconds
Target function gc_1000_5: 123
Passed gc_1000_5: 1
Score: 18
